# Считаю retrieval метрики по DINOv2 

In [4]:
import polars as pl
import numpy as np
from pathlib import Path
import chromadb

In [2]:
PROJECT_ROOT_PATH = Path('/Users/a.r.makarenko/Documents/hse/sneakers-hse')

In [3]:
df = pl.read_parquet(PROJECT_ROOT_PATH / "data/04_dinov2_embeddings.parquet.gzip")

embeddings = np.stack(df["embedding"].to_list()).astype("float32")
labels = np.array(df["class"].to_list())
paths = df["path"].to_list()

In [6]:
client = chromadb.Client()
collection = client.get_or_create_collection(
    "embeddings",
    metadata={"hnsw:space": "cosine"})
collection.add(
    ids=paths,
    embeddings=embeddings,
    metadatas=[{"class": c} for c in labels]
)

In [36]:
def get_neighbors(collection, embeddings, k=10):
    results = collection.query(
        query_embeddings=embeddings,
        n_results=k + 1  # +1 чтобы убрать self
    )['metadatas']
    # print(len(results))
    return np.array([[neighbor['class'] for neighbor in query]
                     for query in results])

def hit_rate_at_k(neighbors, labels, k):
    hits = 0
    for i in range(len(labels)):
        query_label = labels[i]
        retrieved = neighbors[i][1:k+1]
        if query_label in retrieved:
            hits += 1
    return hits / len(labels)

def precision_at_k(neighbors, labels, k):
    precisions = []
    for i in range(len(labels)):
        query_label = labels[i]
        retrieved = neighbors[i][1:k+1]
        relevant = (retrieved == query_label).sum()
        precisions.append(relevant / k)
    return np.mean(precisions)

def average_precision(retrieved, query_label, k):
    score = 0.0
    hits = 0
    for i in range(k):
        if retrieved[i] == query_label:
            hits += 1
            score += hits / (i + 1)
    return score / hits if hits > 0 else 0


def map_at_k(neighbors, labels, k):
    scores = []
    for i in range(len(labels)):
        query_label = labels[i]
        retrieved = neighbors[i]
        scores.append(average_precision(retrieved, query_label, k))
    return np.mean(scores)
neighbors = get_neighbors(collection, embeddings, k=20)
len(neighbors[0])
for k in [1, 5, 10]:
    print(f"\nMetrics @ {k}")
    print("HitRate:", hit_rate_at_k(neighbors, labels, k))
    print("Precision:", precision_at_k(neighbors, labels, k))
    print("MAP:", map_at_k(neighbors, labels, k))


Metrics @ 1
HitRate: 0.9672619047619048
Precision: 0.9672619047619048
MAP: 1.0

Metrics @ 5
HitRate: 0.9895833333333334
Precision: 0.9211309523809523
MAP: 0.9913835152116403

Metrics @ 10
HitRate: 0.9985119047619048
Precision: 0.8836309523809524
MAP: 0.9704646049199
